# 🚌 Bus Number Detection System

This notebook detects bus numbers from a CCTV video using **YOLOv8 + Tesseract OCR**  
and logs results to a live dashboard.

### 👉 After running this notebook, view the live dashboard here:
## 🔴 [bus-monitor.up.railway.app](https://bus-monitor.up.railway.app)
*(replace with your actual Railway URL)*

---

### Steps:
1. Run **Step 1** to install dependencies
2. Run **Step 2** — no changes needed, config is pre-filled
3. Run **Step 3** to download the video from Google Drive
4. Run **Step 4** to upload the model
5. Run **Step 5** to start detection
6. Open the dashboard link above to see results live!

> ⏱️ Detection takes ~5-10 minutes depending on video length

## Step 1 — Install Dependencies

In [ ]:
!pip install -q ultralytics pytesseract mysql-connector-python gdown
!apt-get install -q tesseract-ocr
!mkdir -p ./temp_images
print('✅ All dependencies installed')

## Step 2 — Configuration
> ✅ Pre-filled — no changes needed. Just run this cell.

In [ ]:
# ── Paths ────────────────────────────────────────────────────
VIDEO_PATH     = 'input_video.mp4'
MODEL_PATH     = 'best.pt'
TESSERACT_PATH = '/usr/bin/tesseract'
IMAGES_DIR     = './temp_images/'

# ── Google Drive File ID for the video ───────────────────────
# Video is hosted on Google Drive — no upload needed!
GOOGLE_DRIVE_FILE_ID = 'PASTE_YOUR_VIDEO_FILE_ID_HERE'

# ── Detection settings ───────────────────────────────────────
THRESHOLD         = 0.5
LINE_Y            = 800
VALID_BUS_NUMBERS = [9, 19, 15, 6, 5, 13, 7, 14]
LICENCE_PLATE_MAP = {
    19: 'TN 84 C35619',
     9: 'TN 84 A55709',
    15: 'TN 84 C75915',
     6: 'TN 84 C85806',
     5: 'TN 84 C35805',
    13: 'TN 84 C35913',
     7: 'TN 84 C25697',
    14: 'TN 84 C15514',
}

# ── Railway MySQL (pre-filled, connects to live dashboard) ───
DB_HOST     = 'PASTE_RAILWAY_HOST_HERE'
DB_PORT     = 0000                        # paste your Railway port
DB_USER     = 'root'
DB_PASSWORD = 'PASTE_RAILWAY_PASSWORD_HERE'
DB_NAME     = 'railway'

print('✅ Configuration loaded')

## Step 3 — Download Video from Google Drive

In [ ]:
import gdown, os

print('📥 Downloading video from Google Drive...')
gdown.download(f'https://drive.google.com/uc?id={GOOGLE_DRIVE_FILE_ID}', VIDEO_PATH, quiet=False)
size = os.path.getsize(VIDEO_PATH) / (1024*1024)
print(f'✅ Video downloaded: {size:.1f} MB')

## Step 4 — Upload Model (best.pt)

In [ ]:
import os

if not os.path.exists(MODEL_PATH):
    print('📤 best.pt not found — please upload it now:')
    from google.colab import files
    files.upload()
else:
    size = os.path.getsize(MODEL_PATH) / (1024*1024)
    print(f'✅ Model found: {size:.1f} MB')

## Step 5 — Run Detection
> 🚌 Results will appear in the live dashboard as buses are detected!

In [ ]:
import cv2, datetime, os
import mysql.connector
from ultralytics import YOLO
from pytesseract import pytesseract
from statistics import mode

# ── Setup ────────────────────────────────────────────────────
pytesseract.tesseract_cmd = TESSERACT_PATH
os.makedirs(IMAGES_DIR, exist_ok=True)

# Connect to Railway database
print('🔌 Connecting to database...')
db = mysql.connector.connect(
    host=DB_HOST, port=DB_PORT,
    user=DB_USER, password=DB_PASSWORD,
    database=DB_NAME
)
cursor = db.cursor()
print('✅ Database connected')

# Load video
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
H, W, _ = frame.shape
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'📹 Video: {W}x{H} | {total_frames} frames')

# Load YOLO model
model = YOLO(MODEL_PATH)
print('✅ YOLO model loaded')
print('\n🚌 Starting detection...\n')
print(f'👉 Watch results live: https://bus-monitor.up.railway.app\n')
print('-' * 55)

# ── State ────────────────────────────────────────────────────
l            = []
saved_paths  = []
detected_in  = set()
detected_out = set()
last_detected = 0
frame_num    = 0

# ── Main Loop ────────────────────────────────────────────────
while ret:
    frame_num += 1

    # Progress every 100 frames
    if frame_num % 100 == 0:
        pct = (frame_num / total_frames) * 100
        print(f'  ⏳ Progress: {frame_num}/{total_frames} frames ({pct:.0f}%)', end='\r')

    results = model(frame, verbose=False)[0]

    for result in results.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = result
        if score > THRESHOLD and y1 <= LINE_Y <= y2:
            if len(saved_paths) < 5:
                crop = frame[int(y1):int(y2), int(x1):int(x2)]
                path = os.path.join(IMAGES_DIR, f'crop_{len(saved_paths)}.jpg')
                cv2.imwrite(path, crop)
                saved_paths.append(path)

    if len(saved_paths) == 5:
        for p in saved_paths:
            img  = cv2.imread(p)
            text = pytesseract.image_to_string(
                img, config='-l eng --psm 9 -c tessedit_char_whitelist=1234567890'
            )
            nums = [int(n) for n in text.split() if n.isdigit() and int(n) in VALID_BUS_NUMBERS]
            l.extend(nums)

        saved_paths.clear()
        for f in os.scandir(IMAGES_DIR):
            if f.is_file(): os.remove(f.path)

        if l:
            bus_num = mode(l)
            now     = datetime.datetime.now()
            today   = datetime.date.today()
            plate   = LICENCE_PLATE_MAP.get(bus_num, 'Unknown')

            if bus_num not in detected_in and bus_num not in detected_out:
                print(f'\n  ✅ BUS IN  | #{bus_num} | {plate} | {now.strftime("%H:%M:%S")}')
                cursor.execute(
                    "INSERT INTO bus_number_detection (bus_number, licence_plate_number, In_time, In_date) VALUES (%s,%s,%s,%s)",
                    (bus_num, plate, now, today)
                )
                db.commit()
                detected_in.add(bus_num)
                last_detected = bus_num

            elif bus_num in detected_in and last_detected != bus_num and bus_num not in detected_out:
                print(f'\n  🚌 BUS OUT | #{bus_num} | {now.strftime("%H:%M:%S")}')
                cursor.execute(
                    "UPDATE bus_number_detection SET Out_time=%s, Out_Date=%s WHERE bus_number=%s AND Out_time IS NULL",
                    (now, today, bus_num)
                )
                db.commit()
                detected_out.add(bus_num)
                detected_in.discard(bus_num)
        l.clear()

    ret, frame = cap.read()

# ── Done ─────────────────────────────────────────────────────
cap.release()
cursor.close()
db.close()

print(f'\n\n✅ Detection complete! Processed {frame_num} frames.')
print(f'\n👉 View results on the live dashboard:')
print(f'   https://bus-monitor.up.railway.app')

---
## 👀 View Live Dashboard

Click the link below to see all detected buses with their in/out times:

## 🔴 [Open Live Dashboard](https://bus-monitor.up.railway.app)

The dashboard auto-refreshes every 10 seconds.